In [1]:
import os
import sys
from pathlib import Path

# Ensure src is in sys.path so cnnClassifier imports seamlessly
proj_root = Path(os.getcwd()).resolve()
if proj_root.name == "research":
    proj_root = proj_root.parent
src_path = str(proj_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [2]:
%pwd

'/Users/DELL/Kidney-Disease-classification/research'

In [3]:
# Navigate to repository root safely
if os.path.basename(os.getcwd()) == "research":
    os.chdir("../")


In [4]:
%pwd

'/Users/DELL/Kidney-Disease-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [8]:
from cnnClassifier.utils.common import get_size

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = Path(self.config.local_data_file)
            os.makedirs(zip_download_dir.parent, exist_ok=True)

            if not zip_download_dir.exists() or zip_download_dir.stat().st_size == 0:
                logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")
                file_id = dataset_url.split("/")[-2]
                prefix = "https://drive.google.com/uc?export=download&id="
                gdown.download(prefix + file_id, str(zip_download_dir), quiet=False)
                logger.info(f"Downloaded data into {zip_download_dir}")
            else:
                logger.info(f"Dataset zip file already exists of size: {get_size(zip_download_dir)}")
            return str(zip_download_dir)
        except Exception as e:
            raise e

    def extract_zip_file(self):
        unzip_path = Path(self.config.unzip_dir)
        os.makedirs(unzip_path, exist_ok=True)
        extracted_marker = unzip_path / "data"
        if extracted_marker.exists() and any(extracted_marker.iterdir()):
            logger.info(f"Dataset already extracted at {unzip_path}. Skipping redundant extraction.")
            return
        logger.info(f"Extracting zip file {self.config.local_data_file} into {unzip_path}...")
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extraction completed successfully at {unzip_path}")


In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[INFO: common: YAML file: config/config.yaml loaded successfully]
[INFO: common: YAML file: params.yaml loaded successfully]
[INFO: common: Created directory at: artifacts]
[INFO: common: Created directory at: artifacts/data_ingestion]
[INFO: data_ingestion: Dataset zip file already exists of size: ~ 2236828 KB]
[INFO: data_ingestion: Dataset already extracted at artifacts/data_ingestion. Skipping redundant extraction.]
